# Sesión 9 · LangChain: características útiles y RAG agéntico

**AI Engineer I: Fundamentos de IA Generativa** · Módulo 1 · Edutek

---

## 🎯 Objetivos

Al terminar este notebook podrás:

1. **Recordar las piezas de un agente** en LangChain 1.x (modelo, herramientas, prompt, mensajes, bucle) y leer su traza.
2. **Usar las características que hacen utilizable a un agente**: salida estructurada, streaming, middleware, memoria de corto plazo, contexto de ejecución y manejo de errores en herramientas.
3. **Saber cuándo bajar** de `create_agent` a un grafo propio con LangGraph.
4. **Explicar el RAG agéntico**: qué decisiones pasan al modelo, qué patrones existen, qué cuesta y qué riesgos trae frente a un RAG de flujo fijo y a GraphRAG.

## 🗺️ Mapa

| Parte | Tema | ¿Consume dinero? |
| --- | --- | --- |
| 0 | Preparar el entorno y el modelo de juguete | No |
| 1 | Un agente en una página | No (modelo de juguete) |
| 2 | Mensajes y `content_blocks` | No |
| 3 | Salida estructurada | No |
| 4 | Streaming | No |
| 5 | Middleware | No |
| 6 | Memoria de corto plazo | No |
| 7 | Contexto de ejecución y errores en herramientas | No |
| 8 | LangGraph por debajo y observabilidad | No |
| 9 | RAG agéntico a fondo | No |
| 10 | La práctica: `langchain-agentic-rag` | No con LM Studio |
| 11 | Cierre | — |

> 🧸 **Modelo de juguete.** Para ver el *mecanismo* sin red ni costo, casi todo el código usa un modelo "guionado" que responde con mensajes preparados. Un modelo real decide por sí mismo; el mecanismo que rodea al modelo (herramientas, bucle, middleware, memoria) es **idéntico**. La práctica del final usa un modelo real.

> 📌 **Versiones.** El código sigue la documentación de LangChain **1.x** (probado con `langchain 1.4.9` y `langgraph 1.2.20`, noviembre de 2026). Los tutoriales anteriores a la 1.0 usan interfaces reemplazadas (`AgentExecutor`, `initialize_agent`, `LLMChain`): si un ejemplo las usa, es anterior.

---
# Parte 0 · Preparar el entorno

In [ ]:
import sys

print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), (
    "Esta sesión está fijada a Python 3.12. Ejecuta `uv sync` dentro de la carpeta "
    "s09-langchain-rag-agentico y selecciona el kernel .venv (ver README)."
)

In [ ]:
import warnings
from importlib.metadata import version

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
warnings.filterwarnings("ignore", message=".*v3 streaming protocol.*")

for paquete in ("langchain", "langchain-core", "langgraph"):
    print(f"{paquete:<16}{version(paquete)}")

### El modelo de juguete

`ScriptedModel` hereda de `BaseChatModel` (la clase base de todos los modelos de chat de LangChain) y, en vez de llamar a un proveedor, **devuelve el siguiente mensaje de un guion**. Así vemos el bucle completo, de forma determinista y gratis. Las funciones `llamada` y `final` arman los mensajes del guion: una petición de herramienta y una respuesta de texto.

In [ ]:
from langchain.messages import AIMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult


class ScriptedModel(BaseChatModel):
    """Modelo de juguete: entrega, en orden, los mensajes del guion."""

    script: list
    seen: list = []        # cuántos mensajes recibió en cada llamada
    system_prompts: list = []

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def bind_tools(self, tools, **kwargs):
        return self        # un modelo real recibiría aquí los esquemas de las herramientas

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self.seen.append(len(messages))
        self.system_prompts.append(next((m.content for m in messages if m.type == "system"), ""))
        return ChatResult(generations=[ChatGeneration(message=self.script.pop(0))])


def llamada(herramienta: str, args: dict, id: str) -> AIMessage:
    """Un mensaje del modelo que PIDE una herramienta (no responde texto)."""
    return AIMessage(content="", tool_calls=[{"name": herramienta, "args": args, "id": id}])


def mostrar_traza(mensajes) -> None:
    for m in mensajes:
        if m.type == "human":
            print("👤", m.content)
        elif m.type == "ai" and m.tool_calls:
            for c in m.tool_calls:
                print(f"🤖 pide {c['name']}({c['args']})")
        elif m.type == "ai":
            print("🤖", m.content)
        elif m.type == "tool":
            print("🔧", m.content[:120])

### Interruptor para probar con un modelo real (opcional)

Hay una celda opcional en la Parte 1 que ejecuta el mismo agente con un **modelo real**. Con LM Studio es gratis (usa un modelo de ~12B o más: el de 3B elige mal las herramientas). Con Anthropic u OpenAI cuesta fracciones de centavo y hay que activar el segundo interruptor.

In [ ]:
USAR_MODELO_REAL = False             # ← True para probar la Parte 1 con el modelo de tu .env (LLM_PROVIDER)
EJECUTAR_LLAMADAS_DE_PAGO = False    # ← True solo si LLM_PROVIDER es anthropic u openai (💰 fracciones de centavo)

---
# Parte 1 · Un agente en una página

> **Idea de la sesión anterior, en una línea:** un agente es un **modelo** que, dentro de un **bucle**, decide qué **herramientas** usar hasta resolver la tarea. Esta parte lo recuerda lo justo para que todo lo demás se entienda solo.

## Las piezas

| Pieza | Qué es | En LangChain |
| --- | --- | --- |
| **Modelo** | Quien razona y decide el siguiente paso | Un chat model; `init_chat_model("proveedor:modelo")` o una clase como `ChatAnthropic` |
| **Herramientas** | Funciones que el modelo puede *pedir* que se ejecuten | Funciones con el decorador `@tool`; el docstring es su descripción |
| **Prompt de sistema** | Las reglas del agente | `system_prompt=` |
| **Mensajes** | El estado de la conversación: preguntas, peticiones de herramienta, resultados | `{"messages": [...]}` |
| **Bucle** | Repite *modelo → herramienta → modelo* hasta que el modelo responde sin pedir nada | Lo construye `create_agent` |

**Analogía de infraestructura.** Es un **bucle de reconciliación** (como el de un controlador de Kubernetes): observar, decidir una acción, ejecutarla, volver a observar, hasta que se cumple la condición de salida.

**Lo único que el modelo sabe de una herramienta** es su nombre, su descripción y el esquema de sus parámetros. El modelo **pide**; tu código **ejecuta**.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def service_status(service: str) -> str:
    """Devuelve el estado actual de un servicio de TI (por ejemplo: vpn, correo, wifi)."""
    return f"{service}: operativo"


modelo = ScriptedModel(script=[
    llamada("service_status", {"service": "vpn"}, "c1"),   # el modelo PIDE la herramienta
    AIMessage("La VPN está operativa."),                    # con el resultado a la vista, RESPONDE
])
agente = create_agent(modelo, tools=[service_status], system_prompt="Eres un asistente de la mesa de ayuda.")

resultado = agente.invoke({"messages": [{"role": "user", "content": "¿Cómo está la VPN?"}]})
mostrar_traza(resultado["messages"])

**Qué leer en la traza.** Cuatro mensajes: la pregunta (👤), la petición de herramienta (🤖), el resultado que **tu código** produjo (🔧) y la respuesta final (🤖). El bucle dio **dos vueltas al modelo**: por eso un agente cuesta más de una llamada por pregunta.

### Con un modelo real (opcional)

El agente es el mismo; solo cambia el modelo. `init_chat_model` es la interfaz común: el resto del código no sabe qué proveedor hay detrás.

In [ ]:
# 💰 FYI: con anthropic/openai esta celda consume dinero (≈ US$ 0.001). Con LM Studio (local) es gratis.
import os

from langchain.chat_models import init_chat_model

if USAR_MODELO_REAL:
    proveedor = os.getenv("LLM_PROVIDER", "openai").lower()
    if proveedor != "local" and not EJECUTAR_LLAMADAS_DE_PAGO:
        print("⏭️ Omitida: activa EJECUTAR_LLAMADAS_DE_PAGO para usar un proveedor de pago.")
    else:
        if proveedor == "local":
            real = init_chat_model(os.getenv("LOCAL_LLM_MODEL", "google/gemma-4-12b-qat"), model_provider="openai",
                                   base_url=os.getenv("LOCAL_LLM_URL", "http://localhost:1234/v1"), api_key="lm-studio", temperature=0)
        else:
            real = init_chat_model({"anthropic": "claude-haiku-5-5", "openai": "gpt-6-luna"}[proveedor], model_provider=proveedor)
        agente_real = create_agent(real, tools=[service_status], system_prompt="Eres un asistente de la mesa de ayuda.")
        mostrar_traza(agente_real.invoke({"messages": [{"role": "user", "content": "¿Cómo está la VPN?"}]})["messages"])
else:
    print("(Modelo real desactivado: pon USAR_MODELO_REAL = True para probarlo.)")

---
# Parte 2 · Mensajes y `content_blocks`

Todo en LangChain son **mensajes**; se importan de `langchain.messages`.

| Tipo | Quién lo produce | Qué lleva |
| --- | --- | --- |
| `SystemMessage` | Tú | Las reglas del agente |
| `HumanMessage` | El usuario | La pregunta |
| `AIMessage` | El modelo | Texto y/o **`tool_calls`** (peticiones de herramienta), y `usage_metadata` (tokens) |
| `ToolMessage` | Tu código | El resultado de una herramienta, con el `tool_call_id` de la petición que responde |

### `content_blocks`: un formato común entre proveedores

Cada proveedor devuelve su contenido en su propio formato. La propiedad **`content_blocks`** lo expone como una **lista tipada y estándar**: `text`, `reasoning`, `tool_call`, `image`, `audio`, `file`… Así el código no cambia al cambiar de proveedor.

In [ ]:
for m in resultado["messages"]:
    print(f"{m.type:<6}", m.content_blocks)

In [ ]:
ai_con_herramienta = resultado["messages"][1]
print("tool_calls:", ai_con_herramienta.tool_calls)
print("usage_metadata del modelo de juguete:", ai_con_herramienta.usage_metadata)   # con un modelo real trae input/output/total_tokens

> 💬 **Para pensar:** `usage_metadata` es lo que permite llevar **el costo por pregunta**. En un agente, ¿por qué hay que sumarlo sobre *todas* las llamadas y no solo la última?

---
# Parte 3 · Salida estructurada

Un agente suele terminar con **texto libre**. Para que otro programa lo use (una interfaz, una validación, una base de datos) conviene que termine con **datos con forma conocida**.

## Cómo se pide

Con `response_format=`, el agente devuelve además la clave **`structured_response`**, ya validada contra tu esquema (Pydantic, dataclass, `TypedDict` o JSON Schema).

| Opción | Qué hace | Cuándo |
| --- | --- | --- |
| `response_format=MiEsquema` | LangChain elige: salida nativa del proveedor si existe, o *tool calling* | Lo más simple |
| `ToolStrategy(MiEsquema)` | Modela el esquema como una **herramienta final** que el modelo debe llamar | Funciona con cualquier modelo con *tool calling*; reintenta si el modelo rompe el esquema (`handle_errors`) |
| `ProviderStrategy(MiEsquema)` | Usa la salida estructurada **nativa** del proveedor | Más fiable cuando el modelo la soporta (el proveedor impone el esquema) |

Un esquema JSON "a pelo" debe envolverse en una de las dos estrategias.

**Analogía:** es un **contrato de API**: la respuesta del agente deja de ser "lo que se le ocurrió" y pasa a ser un objeto que se valida antes de entregarse.

In [ ]:
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field


class Respuesta(BaseModel):
    """Respuesta final del asistente."""

    answer: str = Field(description="Respuesta breve en español.")
    citations: list[str] = Field(default_factory=list, description="Ids de los fragmentos usados, p. ej. ['D3'].")
    found: bool = Field(description="False si no se encontró la respuesta.")


modelo = ScriptedModel(script=[
    llamada("service_status", {"service": "correo"}, "c1"),
    # con ToolStrategy, la respuesta final es una "llamada" a una herramienta que se llama como el esquema
    llamada("Respuesta", {"answer": "El correo está operativo.", "citations": [], "found": True}, "c2"),
])
agente = create_agent(modelo, tools=[service_status], response_format=ToolStrategy(Respuesta))

r = agente.invoke({"messages": [{"role": "user", "content": "¿Y el correo?"}]})
print(type(r["structured_response"]).__name__, "→", r["structured_response"])

### Lo que la estructura **no** garantiza

El esquema garantiza la **forma**, no la **verdad**. Un campo `citations` bien formado puede contener ids inventados. Por eso, en la práctica del final, **el código verifica las citas** contra lo que el agente realmente recuperó. *Validar la forma lo hace LangChain; validar el contenido lo haces tú.*

---
# Parte 4 · Streaming

Un agente puede tardar varios pasos. **Streaming** permite mostrar el progreso en lugar de esperar al final.

| Modo (`stream_mode`) | Qué emite | Para qué |
| --- | --- | --- |
| `"updates"` | El **estado tras cada paso** del agente (`model`, `tools`) | Mostrar progreso, depurar |
| `"messages"` | Los **tokens** del modelo, conforme se generan | Escribir la respuesta "en vivo" |
| `"custom"` | Lo que una herramienta emita con `get_stream_writer()` | Avisos de progreso de una herramienta larga |

La documentación actual **recomienda para aplicaciones nuevas** `agent.stream_events(..., version="v3")`, que da iteradores separados por proyección (mensajes, valores, llamadas a herramientas). Su protocolo v3 está marcado como **experimental** en la versión probada.

In [ ]:
modelo = ScriptedModel(script=[llamada("service_status", {"service": "wifi"}, "c1"), AIMessage("El wifi está operativo.")])
agente = create_agent(modelo, tools=[service_status])

for paso in agente.stream({"messages": [{"role": "user", "content": "¿Wifi?"}]}, stream_mode="updates"):
    for nodo, cambio in paso.items():                       # nodo = "model" o "tools"
        print(f"[{nodo}]", [type(m).__name__ for m in cambio["messages"]])

In [ ]:
modelo = ScriptedModel(script=[llamada("service_status", {"service": "wifi"}, "c1"), AIMessage("El wifi está operativo.")])
agente = create_agent(modelo, tools=[service_status])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")        # el protocolo v3 emite un aviso de "experimental"
    flujo = agente.stream_events({"messages": [{"role": "user", "content": "¿Wifi?"}]}, version="v3")
    for tipo, item in flujo.interleave("messages", "tool_calls"):
        if tipo == "tool_calls":
            print(f"🔧 llamada a {item.tool_name}({item.input})")
    print("estado final:", type(flujo.output).__name__, "con", len(flujo.output["messages"]), "mensajes")

---
# Parte 5 · Middleware

## Qué es

**Middleware** es código que se **engancha en puntos del bucle** del agente para observarlo, modificarlo o limitarlo, **sin reescribir el bucle**. Es el mismo patrón que el *middleware* de un servidor web o un *sidecar*: lógica transversal (límites, reintentos, auditoría, seguridad) separada de la lógica de negocio.

## Dónde se engancha

| Gancho | Cuándo corre | Estilo |
| --- | --- | --- |
| `@before_agent` / `@after_agent` | Una vez por invocación | Nodo |
| `@before_model` / `@after_model` | Antes / después de **cada** llamada al modelo | Nodo |
| `@wrap_model_call` | **Envuelve** la llamada al modelo (puedes reintentar, cambiar la petición…) | Envoltura |
| `@wrap_tool_call` | **Envuelve** cada ejecución de herramienta (capturar errores, medir, auditar…) | Envoltura |
| `@dynamic_prompt` | Genera el prompt de sistema en cada llamada | Conveniencia |

También existe la forma de clase (`AgentMiddleware`). Se activan con `create_agent(..., middleware=[...])`.

## Los incluidos

(`from langchain.agents.middleware import …`; confirma los parámetros en la documentación vigente.)

| Middleware | Para qué |
| --- | --- |
| `ToolCallLimitMiddleware` | Tope de llamadas a herramientas por ejecución o por hilo (`run_limit`, `thread_limit`, `tool_name`) |
| `ModelCallLimitMiddleware` | Tope de llamadas al modelo |
| `ToolRetryMiddleware` / `ModelRetryMiddleware` | Reintentos con *backoff* ante fallas |
| `ModelFallbackMiddleware` | Pasar a otro modelo si el primero falla |
| `SummarizationMiddleware` | Resumir la conversación cuando crece (`trigger`, `keep`) |
| `HumanInTheLoopMiddleware` | **Pausar** para aprobación humana antes de una herramienta (exige *checkpointer*) |
| `PIIMiddleware` | Enmascarar o bloquear datos personales (correos, tarjetas…) |
| `ToolErrorMiddleware` | Convertir errores de herramientas en mensajes legibles para el modelo (`langchain>=1.3.14`) |
| `ContextEditingMiddleware`, `LLMToolSelectorMiddleware`, `TodoListMiddleware`… | Gestión de contexto, selección de herramientas, listas de tareas |

### Ejemplo 1 · Un tope para un bucle desbocado

El modelo de juguete insiste en llamar a la herramienta **cinco veces**. Con `run_limit=2`, la tercera y las siguientes **no se ejecutan**: el agente recibe un aviso y continúa.

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

ejecuciones = []


@tool
def buscar(consulta: str) -> str:
    """Busca en el manual."""
    ejecuciones.append(consulta)
    return "sin resultados útiles"


guion = [llamada("buscar", {"consulta": f"intento {i}"}, f"c{i}") for i in range(5)] + [AIMessage("No lo encuentro.")]
agente = create_agent(ScriptedModel(script=guion), tools=[buscar], middleware=[ToolCallLimitMiddleware(tool_name="buscar", run_limit=2)])
agente.invoke({"messages": [{"role": "user", "content": "pregunta imposible"}]}, {"recursion_limit": 40})
print("El modelo pidió 5 búsquedas; se ejecutaron:", len(ejecuciones))

### Ejemplo 2 · Un middleware propio: errores de herramientas

Si una herramienta lanza una excepción, el agente entero puede caerse. Con `@wrap_tool_call` la convertimos en un **`ToolMessage` legible**, y el modelo decide qué hacer (reintentar, avisar). Es el patrón para que *"las fallas sean información para el modelo, no un crash"*.

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage


@tool
def consultar_ticket(ticket_id: str) -> str:
    """Consulta un ticket por su id (formato T-1234)."""
    if not ticket_id.startswith("T-"):
        raise ValueError(f"id inválido: {ticket_id!r}")
    return f"{ticket_id}: abierto"


@wrap_tool_call
def errores_legibles(request, handler):
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(content=f"Error de la herramienta: {e}. Revisa el formato del id e inténtalo de nuevo.",
                           tool_call_id=request.tool_call["id"])


guion = [llamada("consultar_ticket", {"ticket_id": "1234"}, "c1"),       # primer intento: formato malo
         llamada("consultar_ticket", {"ticket_id": "T-1234"}, "c2"),     # el modelo corrige tras leer el error
         AIMessage("El ticket T-1234 está abierto.")]
agente = create_agent(ScriptedModel(script=guion), tools=[consultar_ticket], middleware=[errores_legibles])
mostrar_traza(agente.invoke({"messages": [{"role": "user", "content": "estado del ticket 1234"}]})["messages"])

### Ejemplo 3 · Aprobación humana antes de actuar

`HumanInTheLoopMiddleware` **pausa** el agente antes de ejecutar una herramienta marcada; una persona **aprueba, edita o rechaza**, y el agente continúa. Necesita un *checkpointer* (guarda el estado mientras espera) y un `thread_id`. Es la defensa estándar para herramientas que **modifican** algo.

Las pausas se leen con `version="v2"` (el resultado trae `.interrupts` y `.value`); se reanuda con `Command(resume=...)`.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def cerrar_ticket(ticket_id: str) -> str:
    """Cierra un ticket (acción que MODIFICA datos)."""
    return f"{ticket_id} cerrado"


modelo = ScriptedModel(script=[llamada("cerrar_ticket", {"ticket_id": "T-1"}, "c1"), AIMessage("Listo: ticket cerrado.")])
agente = create_agent(modelo, tools=[cerrar_ticket], checkpointer=InMemorySaver(),
                      middleware=[HumanInTheLoopMiddleware(interrupt_on={"cerrar_ticket": True})])
hilo = {"configurable": {"thread_id": "aprobacion-1"}}

pausa = agente.invoke({"messages": [{"role": "user", "content": "cierra T-1"}]}, hilo, version="v2")
accion = pausa.interrupts[0].value["action_requests"][0]
print("⏸️ El agente pide aprobación para:", accion["name"], accion["args"])

final = agente.invoke(Command(resume={"decisions": [{"type": "approve"}]}), hilo, version="v2")   # una persona aprueba
print("▶️", final.value["messages"][-1].content)

---
# Parte 6 · Memoria de corto plazo

Un agente, por sí solo, **no recuerda** entre invocaciones: cada `invoke` empieza vacío. Para una conversación con preguntas de seguimiento ("¿y si es más grande?") hace falta **memoria de corto plazo**.

## Cómo funciona

| Concepto | Qué es |
| --- | --- |
| **Checkpointer** | Guarda el **estado** del agente (los mensajes) después de cada paso. `InMemorySaver` vive en RAM y sirve para aprender; en producción se usa uno persistente (por ejemplo, Postgres) |
| **`thread_id`** | Identifica la **conversación**. Mismo `thread_id` → continúa; otro → empieza de cero |

**Analogía:** el `thread_id` es el **id de sesión** y el checkpointer es el **almacén de sesiones**.

## Su límite: la ventana de contexto

Todo el historial viaja al modelo en **cada** llamada. Una conversación larga acaba **saturando la ventana de contexto**, encareciendo y degradando las respuestas. Tres remedios, todos con middleware:

| Remedio | Qué hace | Cuidado |
| --- | --- | --- |
| **Recortar** (`@before_model`) | Conserva solo los mensajes recientes | Pierde lo antiguo; no dejes una petición de herramienta sin su resultado |
| **Borrar** (`@after_model`) | Elimina mensajes concretos (`RemoveMessage`) | Idem: el historial debe seguir siendo válido |
| **Resumir** (`SummarizationMiddleware`) | Sustituye lo antiguo por un resumen (`trigger`, `keep`) | Cuesta una llamada al modelo; el resumen puede omitir detalles |

> La memoria de **largo plazo** (preferencias entre conversaciones distintas) es otro tema: usa un *store*, no el historial.

In [ ]:
modelo = ScriptedModel(script=[AIMessage("Hola Ana."), AIMessage("Te llamas Ana."), AIMessage("No sé cómo te llamas.")])
agente = create_agent(modelo, tools=[], checkpointer=InMemorySaver())


def hablar(texto, hilo):
    return agente.invoke({"messages": [{"role": "user", "content": texto}]}, {"configurable": {"thread_id": hilo}})


hablar("Me llamo Ana.", "A")
hablar("¿Cómo me llamo?", "A")        # mismo hilo: ve el historial
hablar("¿Cómo me llamo?", "B")        # hilo distinto: empieza de cero
print("Mensajes que recibió el modelo en cada llamada:", modelo.seen)
print("Estado guardado del hilo A:", len(agente.get_state({"configurable": {"thread_id": "A"}}).values["messages"]), "mensajes")

Las dos primeras llamadas muestran cómo **crece** lo que recibe el modelo (1 mensaje, luego 3: pregunta, respuesta y la nueva pregunta); la del hilo B vuelve a empezar con 1. Ese crecimiento es el **costo oculto** de la memoria.

---
# Parte 7 · Contexto de ejecución y errores en herramientas

## Tres lugares donde vive la información

| Dónde | Qué guarda | Duración | Quién lo escribe |
| --- | --- | --- | --- |
| **Estado** (`state`) | Mensajes y campos propios del agente | La conversación (con checkpointer) | El agente y tus herramientas |
| **Contexto** (`context`) | Datos de **esta ejecución**: usuario, permisos, región, configuración | Solo esa llamada | **Tú**, al invocar |
| **Store** | Memoria de largo plazo | Entre conversaciones | Herramientas y middleware |

El **contexto** se declara con `context_schema=` en `create_agent`, se pasa con `context=` en `invoke`, y se lee en las herramientas con un parámetro **`ToolRuntime`** (que el modelo **no ve**: no es parte del esquema de la herramienta) y en el middleware con `request.runtime`.

**Por qué importa:** el modelo **no debe decidir** a nombre de quién actúa. La identidad y los permisos los inyecta tu código; así una herramienta puede filtrar por usuario sin confiar en el modelo.

In [ ]:
from dataclasses import dataclass

from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain.tools import ToolRuntime


@dataclass
class Contexto:
    usuario: str
    area: str


@tool
def mis_tickets(runtime: ToolRuntime[Contexto]) -> str:
    """Lista los tickets abiertos del usuario que pregunta."""
    return f"Tickets abiertos de {runtime.context.usuario} ({runtime.context.area}): T-17, T-23"


@dynamic_prompt
def prompt_con_contexto(request: ModelRequest) -> str:
    return f"Eres el asistente de la mesa de ayuda. Atiendes a {request.runtime.context.usuario}."


modelo = ScriptedModel(script=[llamada("mis_tickets", {}, "c1"), AIMessage("Tienes dos tickets abiertos: T-17 y T-23.")])
agente = create_agent(modelo, tools=[mis_tickets], middleware=[prompt_con_contexto], context_schema=Contexto)
r = agente.invoke({"messages": [{"role": "user", "content": "¿Qué tickets tengo?"}]}, context=Contexto(usuario="Ana", area="Finanzas"))
mostrar_traza(r["messages"])
print("Prompt de sistema que vio el modelo:", modelo.system_prompts[0])

Fíjate en que la herramienta `mis_tickets` **no recibe parámetros del modelo**: el usuario salió del contexto. El modelo no pudo pedir "los tickets de otra persona".

## Errores en herramientas

Una herramienta **fallará**: red caída, argumento inválido, permiso denegado. Qué hacer:

| Estrategia | Cómo | Cuándo |
| --- | --- | --- |
| **Mensaje legible al modelo** | `@wrap_tool_call` o `ToolErrorMiddleware` (ver Parte 5) | Errores que el modelo puede corregir (formato, parámetro) |
| **Reintentar** | `ToolRetryMiddleware` (con *backoff*) | Fallas transitorias (red, límite de peticiones) |
| **Fallar fuerte** | Dejar que la excepción suba | Errores que no debe "arreglar" el modelo (permisos, integridad) |

---
# Parte 8 · LangGraph por debajo y observabilidad

## `create_agent` es un grafo

`create_agent` construye un **grafo de LangGraph**: un nodo del modelo, un nodo de herramientas y los ganchos del middleware, unidos por aristas condicionales ("¿el modelo pidió una herramienta?"). Verlo ayuda a entender el bucle:

In [ ]:
print(agente.get_graph().draw_mermaid()[:900])

## Cuándo bajar a un grafo propio

| `create_agent` (nivel alto) | `StateGraph` de LangGraph (nivel bajo) |
| --- | --- |
| El modelo decide **todo** el flujo; tú lo acotas con middleware | **Tú** dibujas el flujo: nodos, aristas y ciclos |
| Rápido de escribir; comportamiento "estándar" | Control fino: ramas fijas, pasos obligatorios, varios agentes |
| Cambios de comportamiento → middleware | Cambios de comportamiento → editar el grafo |

**Regla:** empieza con `create_agent`; baja a `StateGraph` cuando necesites que **ciertos pasos ocurran siempre y en cierto orden** (por ejemplo, "después de recuperar, **siempre** evaluar la relevancia") o coordinar varios agentes. El tutorial de RAG agéntico de la documentación de LangGraph está construido a mano con `StateGraph` por esa razón.

## Observabilidad

Un agente es difícil de depurar sin ver su camino. Opciones: la **traza de mensajes** (lo que hemos usado), **middleware propio** que registre tiempos y costos, y **LangSmith**, el servicio de trazas y evaluación de LangChain (opcional; no lo usamos aquí). Lo mínimo útil por pregunta: herramientas pedidas, argumentos, tokens, costo y latencia.

---
# Parte 9 · RAG agéntico a fondo

> **Punto de partida:** ya sabes qué es un RAG (recuperar fragmentos y generar con ellos) y que puede tener mejoras (contexto, fusión, grafos). Aquí cambia **quién controla el flujo**.

## La idea

En un RAG de **flujo fijo**, el código ejecuta siempre los mismos pasos: buscar → armar prompt → generar. En un **RAG agéntico**, la **búsqueda es una herramienta** más del agente: el modelo decide **si** la usa.

## Qué decisiones pasan al modelo

| Decisión | RAG fijo | RAG agéntico |
| --- | --- | --- |
| ¿Hace falta buscar? | Siempre sí | El modelo decide (un saludo o un cálculo no busca) |
| ¿En qué fuente? | Una | Elige entre varias herramientas (manual, grafo, base de datos…) |
| ¿Con qué consulta? | La pregunta tal cual | La que **formula** el modelo (más específica, o una por cada parte) |
| ¿Cuántas veces? | Una | Las que necesite (con tope) |
| ¿Sirve lo recuperado? | No se evalúa | Puede juzgar y **reformular y reintentar** |
| ¿Hay respuesta? | Responde con lo que llegó | Puede decidir que **no la hay** |

## Patrones conocidos

Son ideas descritas en artículos de 2022 a 2024; se citan aquí como **nombres de referencia**, no como recetas exactas:

| Patrón | Idea | Referencia |
| --- | --- | --- |
| **ReAct** | Alternar *razonar → actuar → observar* | Yao et al., 2022 |
| **Adaptive-RAG** | **Enrutar** según la complejidad de la pregunta: responder directo, una búsqueda o varios pasos | Jeong et al., 2024 |
| **Corrective RAG (CRAG)** | **Evaluar** lo recuperado y corregir (reformular, buscar en otra fuente) si no sirve | Yan et al., 2024 |
| **Self-RAG** | El modelo **reflexiona** sobre si necesita recuperar y si su respuesta está respaldada | Asai et al., 2023 |

El tutorial de **RAG agéntico de LangGraph** combina varios: el modelo decide si recuperar → un paso **califica** la relevancia → si no sirve, **reescribe** la pregunta y reintenta → si sirve, genera. Es un grafo con un ciclo.

## Fijo, GraphRAG y agéntico: qué cambia en cada uno

| | RAG fijo | GraphRAG | RAG agéntico |
| --- | --- | --- | --- |
| **Qué cambia** | Nada: es la base | **La estructura** de los datos (grafo de relaciones) | **Quién decide** el flujo (el modelo) |
| **Resuelve mejor** | Preguntas directas | Relaciones y preguntas globales | Fuentes múltiples, preguntas imprevisibles |
| **Costo** | Bajo y predecible | Alto al **indexar** | Alto al **responder** (varias llamadas) |
| **Predecibilidad** | Alta | Alta | Baja: el camino varía |

Se **combinan**: un agente puede tener entre sus herramientas una búsqueda vectorial y una consulta al grafo, y elegir. Regla: **empieza por el más simple que cumpla tu objetivo medido**.

## Costos y latencia

Cada vuelta del bucle es **una llamada al modelo con todo el historial** (que además incluye los fragmentos recuperados). Una pregunta de dos búsquedas suele ser **3 o 4 llamadas** en vez de 1. Se controla con: topes de llamadas a herramientas, `recursion_limit`, modelos económicos para el enrutamiento y resultados de herramienta **cortos**.

## Riesgos propios

| Riesgo | Por qué ocurre | Defensa |
| --- | --- | --- |
| **Inyección indirecta** | Un documento recuperado contiene instrucciones y el modelo las obedece | Tratar lo recuperado como **datos**; mínimo privilegio; sin herramientas que modifiquen sin aprobación |
| **Bucle de búsquedas** | El modelo reformula sin converger | `ToolCallLimitMiddleware`, `recursion_limit`, regla de abstención en el prompt |
| **Citas inventadas** | El modelo cita fragmentos que nunca vio | **Verificar en código** que cada cita apunta a algo recuperado |
| **Respuesta sin buscar** | El modelo "cree saber" y no usa la herramienta | Prompt claro sobre cuándo buscar; medir la elección de herramienta |

## Cómo se evalúa

Se mide por capas, como cualquier RAG, **más** lo propio del agente: ¿**buscó cuando debía** y **no buscó cuando no**? ¿Las **consultas** que formuló eran buenas? ¿Cuántos **pasos** dio y cuánto costó? ¿La respuesta final es **correcta y está respaldada** por citas válidas? Y siempre contra un conjunto fijo de preguntas, con tamaño de muestra honesto.

### Un RAG agéntico mínimo (modelo de juguete)

Un corpus de 5 textos, una herramienta de búsqueda por palabras clave y un agente que **reformula** cuando la primera búsqueda no sirve. El modelo de juguete sigue el guion de lo que haría un modelo real; el mecanismo (herramienta, bucle, verificación de citas) es el verdadero.

In [ ]:
import re

CORPUS = {
    "D1": "La contraseña corporativa vence cada 90 días.",
    "D2": "La VPN solo funciona en Guatemala, México y Colombia.",
    "D3": "Los adjuntos de correo pueden pesar hasta 25 MB.",
    "D4": "Las laptops de préstamo se piden con 2 días hábiles de anticipación.",
    "D5": "Reporta los correos sospechosos con el botón Reportar phishing.",
}
recuperados = {}


@tool
def search_manual(query: str) -> str:
    """Busca en el manual de la mesa de ayuda. Devuelve fragmentos con su id entre corchetes."""
    palabras = {p for p in re.findall(r"\w+", query.lower()) if len(p) > 3}
    hits = {i: t for i, t in CORPUS.items() if palabras & set(re.findall(r"\w+", t.lower()))}
    recuperados.update(hits)
    return "\n".join(f"[{i}] {t}" for i, t in hits.items()) or "Sin resultados."


class RespuestaConCitas(BaseModel):
    """Respuesta final con citas."""

    answer: str
    citations: list[str]
    found: bool


guion = [
    llamada("search_manual", {"query": "conectarme desde Perú"}, "c1"),            # consulta demasiado específica: sin resultados
    llamada("search_manual", {"query": "VPN funciona en qué países"}, "c2"),            # el modelo REFORMULA
    llamada("RespuestaConCitas", {"answer": "La VPN solo funciona en Guatemala, México y Colombia, así que desde Perú no.",
                                  "citations": ["D2", "D9"], "found": True}, "c3"),   # D9 NO existe: una cita inventada
]
agente = create_agent(ScriptedModel(script=guion), tools=[search_manual], response_format=ToolStrategy(RespuestaConCitas),
                      system_prompt="Busca en el manual. Si el resultado no sirve, reformula. Cita los ids.")
r = agente.invoke({"messages": [{"role": "user", "content": "¿Puedo usar la VPN desde Perú?"}]})
mostrar_traza(r["messages"][:-2])

respuesta = r["structured_response"]
invalidas = [c for c in respuesta.citations if c not in recuperados]
print("\nRespuesta:", respuesta.answer)
print("Citas inválidas detectadas por el código:", invalidas or "ninguna")

Dos cosas para notar. **(1)** El agente **reformuló** tras una búsqueda sin resultados: algo que un flujo fijo no hace. **(2)** La verificación en código **atrapó una cita inventada** (`D9`): el modelo siempre puede equivocarse; el sistema no debe depender de que no lo haga.

---
# Parte 10 · La práctica: `langchain-agentic-rag`

La práctica vive en el repositorio **`ai-agents-playground`**, carpeta [`langchain-agentic-rag/`](../../../ai-agents-playground/langchain-agentic-rag/) (en el repositorio, `ai-agents-playground/langchain-agentic-rag`). Es un RAG agéntico completo y pequeño, con un modelo real y un manual ficticio de mesa de ayuda.

| Concepto de este notebook | Dónde está en la práctica |
| --- | --- |
| Modelo, interfaz común (Parte 1) | `model.py` → `build_model()` con `init_chat_model` |
| Herramientas (Parte 1) | `tools.py` → `search_manual`, `list_topics` |
| Prompt de sistema con reglas de cuándo buscar (Parte 1, 9) | `agent.py` → `SYSTEM_PROMPT` |
| Salida estructurada (Parte 3) | `agent.py` → `Answer` + `ToolStrategy` |
| Verificación de citas en código (Parte 3, 9) | `agent.py` → `verify()` |
| Middleware: tope de búsquedas y reintentos (Parte 5) | `agent.py` → `ToolCallLimitMiddleware`, `ToolRetryMiddleware` |
| Memoria de corto plazo (Parte 6) | `agent.py` → `build_checkpointer()`; `main.py --chat` |
| Traza del agente (Parte 1, 8) | `main.py --trace` |
| El mini-RAG que se envuelve como herramienta (Parte 9) | `retriever.py` |

```bash
cd ai-agents-playground/langchain-agentic-rag
cp .env.example .env && uv sync
uv run python main.py --trace "¿Cada cuánto vence mi contraseña?"
uv run python main.py --trace "Olvidé mi contraseña y estoy en casa: ¿cómo la restablezco y cómo me conecto a la red interna?"
uv run python main.py --trace "Hola, ¿cuánto es 15 + 27?"           # no necesita buscar
uv run python main.py --trace "¿Dan soporte a impresoras 3D?"       # fuera de alcance
uv run python main.py --chat --trace                                 # preguntas de seguimiento
```

**Qué observar:** cuándo **no** busca, cuándo hace **dos búsquedas** con consultas distintas, cómo **reformula** y cómo termina con "no lo encuentro" en lugar de inventar.

---
# Parte 11 · Cierre

## Lo que aprendiste

1. Un agente es **modelo + herramientas + bucle**; la traza (👤 🤖 🔧) es la forma de depurarlo.
2. Los **mensajes** son el estado; `content_blocks` da un formato común entre proveedores.
3. La **salida estructurada** (`response_format`) garantiza la **forma**; la **verdad** la verifica tu código.
4. **Middleware** añade límites, reintentos, aprobación humana y privacidad sin reescribir el bucle.
5. La **memoria de corto plazo** es *checkpointer + thread_id*, y su límite es la ventana de contexto.
6. El **contexto de ejecución** lleva identidad y permisos que el modelo no controla.
7. `create_agent` es un grafo de **LangGraph**; baja a un grafo propio cuando necesites pasos obligatorios.
8. Un **RAG agéntico** convierte la búsqueda en herramienta: el modelo decide si busca, qué, cuántas veces y si sirve. Cuesta más y es menos predecible: se acota y se mide.

## ✅ Autoevaluación

1. ¿Qué hace `ToolStrategy` y en qué se diferencia de `ProviderStrategy`?
2. ¿Por qué un agente con `InMemorySaver` "olvida" todo al reiniciar el programa?
3. Un agente con memoria lleva 40 turnos y cada respuesta tarda más y cuesta más. ¿Por qué, y qué dos remedios hay?
4. ¿Qué diferencia hay entre **estado**, **contexto** y **store**? Da un ejemplo de qué guardarías en cada uno.
5. ¿Por qué el parámetro `ToolRuntime` no aparece en el esquema que ve el modelo, y qué ventaja de seguridad da?
6. ¿Cuándo es mejor un `StateGraph` que `create_agent`?
7. Menciona tres decisiones que un RAG agéntico traslada del código al modelo.
8. ¿Qué dos riesgos son más específicos del RAG agéntico y cómo se mitigan?

## 🚀 Retos

- En la Parte 5, cambia `run_limit` a 1 y a 5 y observa cuántas búsquedas se ejecutan.
- Agrega a `search_manual` de la Parte 9 un `ToolRetryMiddleware` y haz que la herramienta falle la primera vez.
- Usa `ModelCallLimitMiddleware` (consulta su documentación) e intenta cortar un bucle del modelo.
- En `langchain-agentic-rag`, añade una herramienta `ticket_status(ticket_id)` simulada y observa cómo el modelo elige entre ella y la búsqueda.

## 📖 Glosario

| Término | Definición |
| --- | --- |
| **Agente** | Modelo que decide, en un bucle, qué herramientas usar |
| **`create_agent`** | Función de LangChain 1.x que construye el agente (un grafo de LangGraph) |
| **Herramienta (`@tool`)** | Función que el modelo puede pedir ejecutar; su docstring es su descripción |
| **`content_blocks`** | Vista estándar y tipada del contenido de un mensaje |
| **Salida estructurada** | Respuesta final validada contra un esquema (`response_format`) |
| **Middleware** | Código que se engancha en el bucle del agente sin reescribirlo |
| **Checkpointer** | Almacén que guarda el estado del agente tras cada paso |
| **`thread_id`** | Identificador de una conversación |
| **Contexto de ejecución** | Datos de una ejecución (usuario, permisos) pasados con `context=` |
| **`ToolRuntime`** | Parámetro oculto que da a una herramienta acceso a estado, contexto y store |
| **LangGraph** | Biblioteca de flujos con estado como grafos, sobre la que corre `create_agent` |
| **LangSmith** | Servicio de trazas y evaluación de LangChain (opcional) |
| **RAG agéntico** | RAG donde la búsqueda es una herramienta y el modelo controla el flujo |
| **Inyección indirecta** | Instrucciones maliciosas escondidas en datos que el modelo lee |

## ➡️ Siguiente

Con esto se cierra la parte de RAG del curso. A continuación se profundiza en **agentes** como sistemas completos: razonamiento, memoria y estándares para conectar herramientas.